In [1]:
# Agent Team Generator - Step-by-step Notebook
# Uses OpenRouter for generation + OpenAI trace

import os
import json
import asyncio
from datetime import datetime
from pathlib import Path
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel
from IPython.display import display, Markdown, HTML
import gradio as gr
from openai import AsyncOpenAI

# OpenAI Agents SDK
from agents import (
    Agent,
    Runner,
    trace,
    handoff,
    set_default_openai_client,
    set_tracing_export_api_key,
    AgentOutputSchema,
)
from agents.mcp import MCPServerStdio
from contextlib import AsyncExitStack

load_dotenv(override=True)

print("✅ Setup complete. OpenRouter + OpenAI Agents SDK ready.")

✅ Setup complete. OpenRouter + OpenAI Agents SDK ready.


In [2]:
class AgentSpec(BaseModel):
    name: str
    role: str
    instructions: str
    handoff_description: str = ""
    tools: List[str] = []
    mcp_servers: List[dict] = []

class TeamSpec(BaseModel):
    manager: AgentSpec
    specialists: List[AgentSpec]
    success_criteria: List[str]
    domain: str
    generated_dir: str

In [3]:

# OpenRouter configuration - uses your API key
openrouter_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost:7860",
        "X-Title": "Agent Generator",
    }
)
set_default_openai_client(openrouter_client)

set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))

# Generator that creates the team (uses OpenRouter)
generator_agent = Agent(
    name="TeamGenerator",
    instructions="""You are an expert at building high-performing teams of AI agents.
Given a prompt, create a complete team with a manager and specialists.
Focus heavily on software engineering when the prompt mentions engineering/manager/dev/team.
Always include clear success criteria.

Return ONLY valid JSON matching the TeamSpec schema.""",
    # OpenRouter model IDs look like "provider/model". The Agents SDK only knows prefixes
    # "openai/" and "litellm/" — so use openai/ + full OpenRouter id (requests go via your OpenRouter client).
    model="openai/openai/gpt-oss-120b",
    # TeamSpec uses nested models + List[dict] for mcp_servers — not OpenAI "strict" schema safe
    output_type=AgentOutputSchema(TeamSpec, strict_json_schema=False),
)

In [4]:
def sanitize_name(name: str) -> str:
    return "".join(c if c.isalnum() else "_" for c in name.lower())

async def generate_team(prompt: str) -> TeamSpec:
    """Generate team spec using OpenRouter with better error handling"""
    print(f"🔄 Generating team for prompt: {prompt[:80]}...")
    
    try:
        with trace("team_generation"):
            result = await Runner.run(generator_agent, prompt)
            
            if result is None:
                raise ValueError("Runner.run() returned None")
            
            if not hasattr(result, 'final_output') or result.final_output is None:
                print("⚠️ No final_output returned. Raw result:", type(result))
                raise ValueError("No structured output received from model")
            
            team: TeamSpec = result.final_output
            
            # Set output directory
            slug = sanitize_name(team.manager.name or "team")
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            team.generated_dir = f"generated_teams/{slug}_{timestamp}"
            
            print(f"✅ Team generated successfully: {team.manager.name}")
            print(f"   Domain: {team.domain}")
            print(f"   Specialists: {len(team.specialists)}")
            return team
            
    except Exception as e:
        print(f"❌ Error generating team: {type(e).__name__}: {e}")
        # Do not return a fake "success" team — that hid failures (e.g. Unknown prefix: anthropic).
        raise


def persist_team(team: TeamSpec) -> str:
    """Write team_spec.json under team.generated_dir (relative to notebook cwd)."""
    base_dir = Path(team.generated_dir)
    base_dir.mkdir(parents=True, exist_ok=True)
    path = base_dir / "team_spec.json"
    path.write_text(team.model_dump_json(indent=2), encoding="utf-8")
    print(f"✅ Team persisted to: {path}")
    return str(base_dir)

In [5]:
def create_agent(spec: AgentSpec, mcp_servers=None):
    """Create real Agent from spec"""
    return Agent(
        name=spec.name,
        instructions=spec.instructions,
        model="openai/openai/gpt-oss-120b",
        handoff_description=spec.handoff_description or f"Transfer to {spec.name}",
    )

async def build_team(team_spec: TeamSpec):
    """Build manager with handoffs to all specialists."""
    specialists = [create_agent(s) for s in team_spec.specialists]
    mgr_instructions = team_spec.manager.instructions
    if not specialists:
        mgr_instructions += (
            "\n\nYou have no specialist agents to hand off to. Answer the user's task directly "
            "using your own expertise."
        )

    manager = Agent(
        name=team_spec.manager.name,
        instructions=mgr_instructions,
        handoffs=specialists,
        model="openai/openai/gpt-oss-120b",
    )
    
    return manager, specialists

In [6]:
async def run_team_with_evaluation(team_spec: TeamSpec, task: str):
    """Run the team on a task and evaluate against success criteria"""
    manager, _ = await build_team(team_spec)
    
    async with AsyncExitStack() as stack:
        # Add MCP servers here if needed
        # Example: mcp_server = await stack.enter_async_context(MCPServerStdio(...))
        
        with trace("team_execution"):
            result = await Runner.run(manager, task)
    
    output = result.final_output
    
    # Simple evaluation using OpenRouter
    eval_prompt = f"""Task: {task}
Output: {output}
Success Criteria: {team_spec.success_criteria}

Rate how well this output meets the success criteria (0-100).
Return only a JSON with keys: score, passed, feedback"""

    eval_result = await openrouter_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": eval_prompt}],
        response_format={"type": "json_object"}
    )
    
    eval_data = json.loads(eval_result.choices[0].message.content)
    
    return {
        "output": output,
        "evaluation": eval_data,
        "success_criteria": team_spec.success_criteria,
        "trace_id": result.trace_id if hasattr(result, 'trace_id') else None
    }

In [ ]:
def create_gradio_interface():
    with gr.Blocks(title="Agent Team Generator") as demo:
        gr.Markdown(
            "# 🤖 AI Agent Team Generator\n"
            "Generate a team from your prompt, then run a task and score it against success criteria.\n\n"
            "**Models:** use `openai/<openrouter-model-id>` in Agents SDK (e.g. `openai/openai/gpt-oss-120b` for model id `openai/gpt-oss-120b`) "
            "so calls go through your OpenRouter client."
        )

        last_team_state = gr.State(value=None)

        with gr.Tab("Generate Team"):
            prompt_input = gr.Textbox(
                label="Describe the team you want",
                placeholder="You are an engineering manager. Spawn a team of engineers to build a full-stack web application with user authentication.",
                lines=5,
                value="You are an engineering manager, spawn a team of engineers that would work on an engineering task.",
            )
            generate_btn = gr.Button("Generate Team", variant="primary")

            output_md = gr.Markdown()
            team_output = gr.JSON(label="Team JSON")

            async def generate(prompt: str):
                if not prompt or not str(prompt).strip():
                    return "**Error:** Please enter a prompt.", {"error": "Empty prompt"}, None

                try:
                    team = await generate_team(prompt)
                    persist_team(team)
                    dumped = team.model_dump()

                    md = f"""
**✅ Team generated**

**Manager:** {team.manager.name}
**Domain:** {team.domain}
**Specialists:** {len(team.specialists)}
**Saved to:** `{team.generated_dir}/team_spec.json`

**Success criteria:**
""" + "\n".join([f"- {c}" for c in team.success_criteria])

                    return md, dumped, dumped
                except Exception as e:
                    import traceback

                    error_msg = f"**Error:** `{type(e).__name__}`: {e}"
                    print("Full traceback:")
                    traceback.print_exc()
                    return error_msg, {"error": str(e), "type": type(e).__name__}, None

            generate_btn.click(
                fn=generate,
                inputs=[prompt_input],
                outputs=[output_md, team_output, last_team_state],
            )

        with gr.Tab("Run & Evaluate"):
            gr.Markdown("Uses the last generated team (manager + specialist handoffs) and `run_team_with_evaluation`.")
            task_input = gr.Textbox(
                label="Task for the team",
                lines=4,
                placeholder="e.g. Draft a 1-page plan for auth, API, and deployment for the web app.",
            )
            run_btn = gr.Button("Run task & evaluate", variant="primary")
            eval_output = gr.Markdown()

            async def run_eval(task: str, team_dict):
                if not team_dict:
                    return "## No team loaded\nGenerate a team in the **Generate Team** tab first."
                if not task or not str(task).strip():
                    return "## Enter a task\nDescribe what you want the team to produce."

                try:
                    team = TeamSpec.model_validate(team_dict)
                    result = await run_team_with_evaluation(team, str(task).strip())
                    ev = result.get("evaluation") or {}
                    out = result.get("output", "")
                    out_preview = str(out)[:6000]
                    return f"""## Model output

{out_preview}

---

## Evaluation vs success criteria

- **Score:** {ev.get("score", "n/a")}
- **Passed:** {ev.get("passed", "n/a")}
- **Feedback:** {ev.get("feedback", "n/a")}
"""
                except Exception as e:
                    import traceback

                    traceback.print_exc()
                    return f"**Error:** `{type(e).__name__}`: {e}"

            run_btn.click(
                fn=run_eval,
                inputs=[task_input, last_team_state],
                outputs=[eval_output],
            )

    return demo

# Launch with simpler settings
demo = create_gradio_interface()
demo.launch(
    inbrowser=True,
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


🔄 Generating team for prompt: You are an engineering manager, spawn a team of engineers that would work on an ...
❌ Error generating team: UserError: Unknown prefix: anthropic
✅ Team persisted to: generated_teams/debug_20260328_1758/team_spec.json
